# 🏈 TensorCraft Playbook: CIFAR-10 Benchmark with Keras

This notebook is intentionally simple and tolerant.

It does **not** stop execution when a Colab TPU runtime is detected but TensorFlow cannot initialize it as a TPU. Instead, it records the requested runtime, the TensorFlow-visible device, and the effective strategy used for training.

Use this version when Colab allocates a TPU runtime but TensorFlow sees the environment as GPU or CPU after installing the required libraries.


## Step 0 — Optional dependency installation

Run this cell only if TensorFlow is missing in the Colab runtime.

After installation, manually restart the runtime using:

`Runtime > Restart runtime`

Then continue from Step 1. Do not run this cell repeatedly.


In [ ]:
# ============================================================
# STEP 0 - OPTIONAL DEPENDENCY INSTALLATION
# Run only if TensorFlow is missing.
# After this cell finishes, manually restart the runtime.
# ============================================================

RUN_INSTALL = True  # Change to True only when TensorFlow is missing.

if RUN_INSTALL:
    import sys
    import subprocess

    packages = [
        "ml_dtypes>=0.5.1",
        "tensorstore>=0.1.82",
        "jax>=0.6.0",
        "jaxlib>=0.6.0",
        "tensorflow>=2.19.0",
    ]

    for package in packages:
        print(f"Installing {package} ...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            package,
        ])

    print("Installation finished. Now manually restart the runtime before continuing.")
else:
    print("Skipping installation. Set RUN_INSTALL=True only if TensorFlow is missing.")


Installing ml_dtypes>=0.5.1 ...
Installing tensorstore>=0.1.82 ...


## Step 1 — Environment and import validation


In [ ]:
# ============================================================
# STEP 1 - ENVIRONMENT AND IMPORT VALIDATION
# ============================================================

import os
import sys
import time
import json
import platform
import importlib
from datetime import datetime

modules = ["ml_dtypes", "jax", "jaxlib", "tensorstore", "tensorflow"]
module_versions = {}

print("=== Python Environment ===")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("COLAB_TPU_ADDR:", os.environ.get("COLAB_TPU_ADDR"))
print("TPU_NAME:", os.environ.get("TPU_NAME"))

print()
print("=== Import Validation ===")
for module_name in modules:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "version attribute not available")
        module_versions[module_name] = version
        print(f"{module_name}: IMPORT OK - {version}")
    except Exception as e:
        module_versions[module_name] = f"IMPORT FAILED: {repr(e)}"
        print(f"{module_name}: IMPORT FAILED")
        print("Error:", repr(e))

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print()
print("TensorFlow:", tf.__version__)


## Step 2 — Device detection without blocking execution

This cell tries TPU initialization, but it does not fail the notebook if TPU resolver fails. If TensorFlow sees a GPU, the training will run as GPU and the result will be labeled as such.


In [ ]:
# ============================================================
# STEP 2 - DEVICE DETECTION WITHOUT BLOCKING EXECUTION
# ============================================================

REQUESTED_RUNTIME = "COLAB_TPU_RUNTIME"  # For logging only. Options: CPU_RUNTIME, GPU_RUNTIME, COLAB_TPU_RUNTIME

physical_cpus = tf.config.list_physical_devices("CPU")
physical_gpus = tf.config.list_physical_devices("GPU")
physical_tpus = tf.config.list_physical_devices("TPU")

tpu_resolver_status = {
    "attempted": True,
    "success": False,
    "error": None,
    "cluster_spec": None,
}

strategy = None
effective_hardware = None
verified_tpu = False

print("=== TensorFlow Physical Devices ===")
print("CPUs:", physical_cpus)
print("GPUs:", physical_gpus)
print("TPUs before resolver:", physical_tpus)

# Try TPU first, but do not block execution if it fails.
try:
    print()
    print("Trying TPUClusterResolver...")
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tpu_resolver_status["cluster_spec"] = resolver.cluster_spec().as_dict()

    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)

    effective_hardware = "TPU"
    verified_tpu = True
    tpu_resolver_status["success"] = True

    print("TPU initialized successfully.")

except Exception as e:
    tpu_resolver_status["error"] = repr(e)
    print("TPU resolver/initialization failed, but execution will continue.")
    print("Resolver error:", repr(e))

    # If TPU is not usable, use what TensorFlow actually sees.
    physical_gpus = tf.config.list_physical_devices("GPU")

    if physical_gpus:
        try:
            for gpu in physical_gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as memory_error:
            print("GPU memory growth could not be configured after TensorFlow initialization:", memory_error)

        strategy = tf.distribute.MirroredStrategy()
        effective_hardware = "GPU"
    else:
        strategy = tf.distribute.get_strategy()
        effective_hardware = "CPU"

print()
print("=== Effective Execution Strategy ===")
print("Requested runtime:", REQUESTED_RUNTIME)
print("Effective hardware:", effective_hardware)
print("Strategy:", type(strategy).__name__)
print("Replicas:", strategy.num_replicas_in_sync)
print("Verified TPU:", verified_tpu)
print("Logical TPUs:", tf.config.list_logical_devices("TPU"))
print("Logical GPUs:", tf.config.list_logical_devices("GPU"))


## Step 3 — Smoke test on the effective device


In [ ]:
# ============================================================
# STEP 3 - SMOKE TEST
# ============================================================

with strategy.scope():
    a = tf.ones((1024, 1024))
    b = tf.ones((1024, 1024))
    c = tf.matmul(a, b)

print("Smoke test OK.")
print("Result shape:", c.shape)
print("Tensor device:", c.device)


## Step 4 — CIFAR-10 data pipeline


In [ ]:
# ============================================================
# STEP 4 - CIFAR-10 DATA PIPELINE
# ============================================================

BATCH_SIZE_PER_REPLICA = 64
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
EPOCHS = 5

print("Global batch size:", GLOBAL_BATCH_SIZE)
print("Epochs:", EPOCHS)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

train_ds = (
    train_ds
    .shuffle(10000)
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset ready.")


## Step 5 — Keras CNN model


In [ ]:
# ============================================================
# STEP 5 - KERAS CNN MODEL
# ============================================================

def build_model():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax"),
    ])
    return model

with strategy.scope():
    model = build_model()
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

model.summary()


## Step 6 — Training and benchmark logging


In [ ]:
# ============================================================
# STEP 6 - TRAINING AND BENCHMARK LOGGING
# ============================================================

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
)

total_time = time.time() - start_time
images_processed = len(x_train) * EPOCHS
throughput_images_per_second = images_processed / total_time

final_train_accuracy = float(history.history["accuracy"][-1])
final_val_accuracy = float(history.history["val_accuracy"][-1])

benchmark_result = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "requested_runtime": REQUESTED_RUNTIME,
    "effective_hardware": effective_hardware,
    "strategy": type(strategy).__name__,
    "replicas": strategy.num_replicas_in_sync,
    "verified_tpu": verified_tpu,
    "tpu_resolver_status": tpu_resolver_status,
    "tensorflow_version": tf.__version__,
    "module_versions": module_versions,
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "epochs": EPOCHS,
    "total_time_seconds": total_time,
    "throughput_images_per_second": throughput_images_per_second,
    "final_train_accuracy": final_train_accuracy,
    "final_val_accuracy": final_val_accuracy,
}

print()
print("=== Benchmark Result ===")
print(json.dumps(benchmark_result, indent=2))


Epoch 1/5
781/781 ━━━━━━━━━━━━━━━━━━━━ 78s 98ms/step - accuracy: 0.4334 - loss: 1.5579 - val_accuracy: 0.5740 - val_loss: 1.1692
Epoch 2/5
781/781 ━━━━━━━━━━━━━━━━━━━━ 76s 98ms/step - accuracy: 0.5879 - loss: 1.1626 - val_accuracy: 0.6561 - val_loss: 0.9725
Epoch 3/5
781/781 ━━━━━━━━━━━━━━━━━━━━ 77s 99ms/step - accuracy: 0.6515 - loss: 0.9906 - val_accuracy: 0.6915 - val_loss: 0.8825
Epoch 4/5
781/781 ━━━━━━━━━━━━━━━━━━━━ 75s 97ms/step - accuracy: 0.6915 - loss: 0.8752 - val_accuracy: 0.7051 - val_loss: 0.8640
Epoch 5/5
781/781 ━━━━━━━━━━━━━━━━━━━━ 80s 95ms/step - accuracy: 0.7251 - loss: 0.7882 - val_accuracy: 0.7323 - val_loss: 0.7852

=== Benchmark Result ===
{
  "timestamp": "2026-05-01T03:48:08.424616Z",
  "requested_runtime": "COLAB_TPU_RUNTIME",
  "effective_hardware": "CPU",
  "strategy": "_DefaultDistributionStrategy",
  "replicas": 1,
  "verified_tpu": false,
  "tpu_resolver_status": {
    "attempted": true,
    "success": false,
    "error": "ValueError('Please provide a TPU

/tmp/ipykernel_1418/594443921.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


## Step 7 — Export results

The result file makes it explicit whether the run was a true TPU execution or a TPU runtime that TensorFlow executed as GPU/CPU.


In [ ]:
# ============================================================
# STEP 7 - EXPORT RESULTS
# ============================================================

import pandas as pd

safe_hardware_name = effective_hardware.lower()
result_json_path = f"tensorcraft_cifar10_result_{safe_hardware_name}.json"
result_csv_path = f"tensorcraft_cifar10_result_{safe_hardware_name}.csv"

with open(result_json_path, "w") as f:
    json.dump(benchmark_result, f, indent=2)

pd.DataFrame([benchmark_result]).to_csv(result_csv_path, index=False)

print("Saved:", result_json_path)
print("Saved:", result_csv_path)


Saved: tensorcraft_cifar10_result_cpu.json
Saved: tensorcraft_cifar10_result_cpu.csv


## Interpretation

If `requested_runtime` is `COLAB_TPU_RUNTIME` but `effective_hardware` is `GPU` or `CPU`, the benchmark is valid for the device TensorFlow actually used, not as a TPU benchmark.

This is the desired behavior for this experiment: run the workload, collect the result, and document the mismatch between the allocated Colab runtime and the TensorFlow-visible execution device.
